In [1]:
import os
import time
import pandas as pd
import numpy as np

# =========================
# 유틸: 로그 / 시간 / 메모리
# =========================
T0 = time.time()

def log(msg):
    elapsed = time.time() - T0
    print(f"[{elapsed:8.1f}s] {msg}", flush=True)

def df_info(name, df):
    mem_mb = df.memory_usage(deep=True).sum() / (1024**2)
    log(f"{name}: shape={df.shape}, memory≈{mem_mb:,.1f} MB")

# =========================
# 경로 설정 (로컬 환경에 맞게 수정)
# =========================
BASE_DIR = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files"
OUT_DIR = os.path.join(BASE_DIR, "analysis_output")

df2_path = r"C:/Users/qkrtl/Downloads/df2.csv"

cluster_paths = [
    os.path.join(OUT_DIR, "erangel_clustered.parquet"),
    os.path.join(OUT_DIR, "miramar_clustered.parquet"),
    os.path.join(OUT_DIR, "taego_clustered.parquet"),
    os.path.join(OUT_DIR, "rondo_clustered.parquet"),
]

os.makedirs(OUT_DIR, exist_ok=True)

log("시작")

# =========================
# 1) parquet(클러스터링 결과) 모두 로드
# =========================
cluster_frames = []
for i, p in enumerate(cluster_paths, 1):
    if os.path.exists(p):
        t1 = time.time()
        log(f"[{i}/{len(cluster_paths)}] parquet 로드 시작: {os.path.basename(p)}")
        df = pd.read_parquet(p)
        df["__source_parquet"] = os.path.basename(p)  # 출처 추적용
        cluster_frames.append(df)
        log(f"[{i}/{len(cluster_paths)}] parquet 로드 완료: {os.path.basename(p)} ({time.time()-t1:.1f}s)")
        df_info(os.path.basename(p), df)
    else:
        log(f"[{i}/{len(cluster_paths)}] SKIP 없음: {p}")

if not cluster_frames:
    raise FileNotFoundError("clustered parquet 파일을 찾지 못했습니다.")

log("parquet concat 시작")
t1 = time.time()
df_parquet_all = pd.concat(cluster_frames, ignore_index=True)
log(f"parquet concat 완료 ({time.time()-t1:.1f}s)")
df_info("df_parquet_all", df_parquet_all)

# 메모리 절약: 더 이상 개별 프레임 필요 없으면 해제
del cluster_frames

# =========================
# 2) df2.csv 로드
# =========================
log("df2.csv 로드 시작")
t1 = time.time()
df2 = pd.read_csv(df2_path)
log(f"df2.csv 로드 완료 ({time.time()-t1:.1f}s)")
df_info("df2", df2)

# 키 컬럼 맞추기
if "playerId" in df2.columns and "accountId" not in df2.columns:
    df2 = df2.rename(columns={"playerId": "accountId"})
    log("df2: playerId -> accountId rename")

# 데이터 타입 통일 (키 mismatch 방지)
for c in ["matchId", "accountId"]:
    if c in df_parquet_all.columns:
        df_parquet_all[c] = df_parquet_all[c].astype(str)
    if c in df2.columns:
        df2[c] = df2[c].astype(str)
log("조인 키 타입(str) 통일 완료")

# =========================
# 3) 조인 전 키 중복 확인
# =========================
KEYS = ["matchId", "accountId"]

missing_left = [c for c in KEYS if c not in df_parquet_all.columns]
missing_right = [c for c in KEYS if c not in df2.columns]
if missing_left or missing_right:
    raise KeyError(f"조인키 누락 - parquet:{missing_left}, df2:{missing_right}")

log("중복키 검사 시작")
t1 = time.time()
dup_left = df_parquet_all.duplicated(subset=KEYS).sum()
dup_right = df2.duplicated(subset=KEYS).sum()
log(f"중복키 검사 완료 ({time.time()-t1:.1f}s)")
log(f"[중복키] parquet={dup_left}, df2={dup_right}")

if dup_right > 0:
    log("df2 중복키 행 추출/저장 시작")
    dup_df2_rows = df2[df2.duplicated(subset=KEYS, keep=False)].copy()
    dup_df2_path = os.path.join(OUT_DIR, "df2_duplicate_keys_debug.csv")
    dup_df2_rows.to_csv(dup_df2_path, index=False, encoding="utf-8-sig")
    log(f"[DEBUG] 저장 완료: {dup_df2_path} ({dup_df2_rows.shape})")

    before = df2.shape
    df2 = df2.drop_duplicates(subset=KEYS, keep="last").copy()
    log(f"df2 dedupe 완료: {before} -> {df2.shape}")

if dup_left > 0:
    log("parquet 중복키 행 추출/저장 시작")
    dup_parquet_rows = df_parquet_all[df_parquet_all.duplicated(subset=KEYS, keep=False)].copy()
    dup_pq_path = os.path.join(OUT_DIR, "parquet_duplicate_keys_debug.csv")
    dup_parquet_rows.to_csv(dup_pq_path, index=False, encoding="utf-8-sig")
    log(f"[DEBUG] 저장 완료: {dup_pq_path} ({dup_parquet_rows.shape})")

# =========================
# 4) 모든 컬럼 유지 merge
# =========================
log("merge 시작 (이 단계는 행 수가 크면 오래 걸릴 수 있음)")
t1 = time.time()
df_master = df_parquet_all.merge(
    df2,
    on=KEYS,
    how="left",
    suffixes=("_pq", "_df2")
)
log(f"merge 완료 ({time.time()-t1:.1f}s)")
df_info("df_master", df_master)

# 조인율 체크
probe_candidates = ["tier", "servername", "currentRankPoint", "game_type"]
probe = next((c for c in probe_candidates if c in df_master.columns), None)
if probe:
    log(f"[조인율] {probe}: {df_master[probe].notna().mean():.2%}")

# =========================
# 5) 컬럼 목록 저장 (가벼움)
# =========================
log("컬럼 목록/분류 저장 시작")
t1 = time.time()

pd.DataFrame({"column_name": df_master.columns}).to_csv(
    os.path.join(OUT_DIR, "master_all_columns_list.csv"),
    index=False,
    encoding="utf-8-sig"
)

pq_cols = set(df_parquet_all.columns)
df2_cols = set(df2.columns)
common_cols = sorted(list(pq_cols & df2_cols))
pq_only_cols = sorted(list(pq_cols - df2_cols))
df2_only_cols = sorted(list(df2_cols - pq_cols))

pd.DataFrame({"common_columns": pd.Series(common_cols)}).to_csv(
    os.path.join(OUT_DIR, "columns_common_parquet_df2.csv"), index=False, encoding="utf-8-sig"
)
pd.DataFrame({"parquet_only_columns": pd.Series(pq_only_cols)}).to_csv(
    os.path.join(OUT_DIR, "columns_parquet_only.csv"), index=False, encoding="utf-8-sig"
)
pd.DataFrame({"df2_only_columns": pd.Series(df2_only_cols)}).to_csv(
    os.path.join(OUT_DIR, "columns_df2_only.csv"), index=False, encoding="utf-8-sig"
)

log(f"컬럼 목록/분류 저장 완료 ({time.time()-t1:.1f}s)")

# =========================
# 6) 저장 전략 선택
# =========================
# (A) 샘플 먼저 저장해서 CSV 저장 속도 감 잡기
sample_path = os.path.join(OUT_DIR, "master_all_columns_sample_5000.csv")
log("샘플 CSV 저장 시작 (5000행)")
t1 = time.time()
df_master.head(5000).to_csv(sample_path, index=False, encoding="utf-8-sig")
log(f"샘플 CSV 저장 완료 ({time.time()-t1:.1f}s): {sample_path}")

# (B) 전체 CSV 저장 (제일 오래 걸릴 수 있음)
master_csv = os.path.join(OUT_DIR, "master_all_columns.csv")
log("전체 CSV 저장 시작 (매우 오래 걸릴 수 있음)")
t1 = time.time()
df_master.to_csv(master_csv, index=False, encoding="utf-8-sig")
log(f"전체 CSV 저장 완료 ({time.time()-t1:.1f}s): {master_csv}")

log("전체 작업 완료")

[     0.0s] 시작
[     0.0s] [1/4] parquet 로드 시작: erangel_clustered.parquet
[     0.2s] [1/4] parquet 로드 완료: erangel_clustered.parquet (0.2s)
[     0.2s] erangel_clustered.parquet: shape=(234245, 38), memory≈94.6 MB
[     0.2s] [2/4] parquet 로드 시작: miramar_clustered.parquet
[     0.2s] [2/4] parquet 로드 완료: miramar_clustered.parquet (0.0s)
[     0.2s] miramar_clustered.parquet: shape=(193518, 32), memory≈71.2 MB
[     0.2s] [3/4] parquet 로드 시작: taego_clustered.parquet
[     0.3s] [3/4] parquet 로드 완료: taego_clustered.parquet (0.1s)
[     0.3s] taego_clustered.parquet: shape=(245900, 32), memory≈89.8 MB
[     0.3s] [4/4] parquet 로드 시작: rondo_clustered.parquet
[     0.3s] [4/4] parquet 로드 완료: rondo_clustered.parquet (0.0s)
[     0.3s] rondo_clustered.parquet: shape=(177771, 32), memory≈65.1 MB
[     0.3s] parquet concat 시작
[     0.4s] parquet concat 완료 (0.0s)
[     0.4s] df_parquet_all: shape=(851434, 39), memory≈350.8 MB
[     0.4s] df2.csv 로드 시작
[     3.3s] df2.csv 로드 완료 (3.0s)
[     3.3s]

In [3]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df = pd.read_csv("C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/master_all_columns.csv")

In [5]:
df.isnull().sum()
df.T

,0,1,2,3,4,5,6,7,8,9,...,851424,851425,851426,851427,851428,851429,851430,851431,851432,851433
matchId,000c936c-b00b-47f0-9e1b-493d589f99bd,000c936c-b00b-47f0-9e1b-493d589f99bd,000c936c-b00b-47f0-9e1b-493d589f99bd,000c936c-b00b-47f0-9e1b-493d589f99bd,000c936c-b00b-47f0-9e1b-493d589f99bd,000c936c-b00b-47f0-9e1b-493d589f99bd,000c936c-b00b-47f0-9e1b-493d589f99bd,000c936c-b00b-47f0-9e1b-493d589f99bd,000c936c-b00b-47f0-9e1b-493d589f99bd,000c936c-b00b-47f0-9e1b-493d589f99bd,...,ffe66b7e-9ca1-4580-b123-71e413fb1bc7,ffe66b7e-9ca1-4580-b123-71e413fb1bc7,ffe66b7e-9ca1-4580-b123-71e413fb1bc7,ffe66b7e-9ca1-4580-b123-71e413fb1bc7,ffe66b7e-9ca1-4580-b123-71e413fb1bc7,ffe66b7e-9ca1-4580-b123-71e413fb1bc7,ffe66b7e-9ca1-4580-b123-71e413fb1bc7,ffe66b7e-9ca1-4580-b123-71e413fb1bc7,ffe66b7e-9ca1-4580-b123-71e413fb1bc7,ffe66b7e-9ca1-4580-b123-71e413fb1bc7
accountId,account.0038d4bb0dcc4a05b68d39589da0236e,account.0616e81ecec241c2915f1bbc52b6958e,account.0901bd6b4a6942d894cddab5aa0907f5,account.0d2472cda90944c69b0111471bed1af2,account.10cfdd7235c54acab7252a396d392466,account.11347af861ef41baa255a9f11990a300,account.13e58593efb440f68433359c47b982fd,account.148814d1227a448a9344ffdc26e5168a,account.1583f1ee7dcc4bbdb764aa2d251a89ed,account.16caa74dce934511987bc2d684b48241,...,account.d2b23e72cb014200801505227d0f0fd0,account.d522bde107284a57a90bf6593c8d71b3,account.dafbb2f7c067478ca2ead8de7072ba27,account.dba84b4d376845baa4213ae96e86f78e,account.e1e55b409e0d41de86b119c1d228f5b0,account.e900a5a5e99048589c5cd5ff6f5c9557,account.e901040855f440d0bef4083161b520f3,account.e965fcccfadc4afa91da042e7a816734,account.ebf76ba6c320422295448c5a02e1cd60,account.f41953d8ca164d2a9692e7cf21130c10
rotation_timing_score,0.54413,0.566873,0.54413,0.54413,0.587462,0.494048,0.525425,0.619443,0.652191,0.629495,...,0.54413,0.54413,0.54413,0.54413,0.54413,0.54413,0.54413,0.54413,0.54413,0.54413
vehicle_use_ratio,0.0,0.065359,0.0,0.0,0.24183,0.352273,0.237288,0.338028,0.058824,0.18,...,0.048193,0.178571,0.0,0.296774,0.176991,0.188679,0.0,0.169231,0.065217,0.10241
bluezone_exposure_ratio,0.0,0.176471,0.0,0.0,0.0,0.363636,0.0,0.0,0.058824,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.184615,0.358696,0.012048
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
winPlace_df2,15.0,2.0,14.0,7.0,3.0,10.0,13.0,12.0,7.0,13.0,...,2.0,4.0,11.0,3.0,7.0,12.0,16.0,5.0,8.0,1.0
servername,steam,steam,steam,steam,steam,steam,steam,steam,steam,steam,...,steam,steam,steam,steam,steam,steam,steam,steam,steam,steam
currentRankPoint,971.0,2460.0,57.0,2425.0,3495.0,1072.0,3580.0,4033.0,2166.0,0.0,...,2280.0,3281.0,1830.0,1945.0,2926.0,2587.0,1895.0,985.0,1097.0,1717.0
game_type,competitive,competitive,competitive,competitive,competitive,competitive,competitive,competitive,competitive,competitive,...,competitive,competitive,competitive,competitive,competitive,competitive,competitive,competitive,competitive,competitive


In [ ]:
## validation(검증)
# merge ( 맵 이름_clustered.parquet <> df2 )
# 1. master가 parquet의 모든 키를 포함하는지
# 2. df2가 보조로 붙은 left join처럼 보이는지

import os
import pandas as pd

# =========================
# 경로 설정
# =========================
BASE_DIR = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files"
OUT_DIR = os.path.join(BASE_DIR, "analysis_output")

master_path = os.path.join(OUT_DIR, "master_all_columns.csv")
df2_path = r"C:/Users/qkrtl/Downloads/df2.csv"

cluster_paths = {
    "erangel": os.path.join(OUT_DIR, "erangel_clustered.parquet"),
    "miramar": os.path.join(OUT_DIR, "miramar_clustered.parquet"),
    "taego":   os.path.join(OUT_DIR, "taego_clustered.parquet"),
    "rondo":   os.path.join(OUT_DIR, "rondo_clustered.parquet"),
}

KEYS = ["matchId", "accountId"]

# =========================
# 0) 로드
# =========================
print("=== LOAD ===")
df_master = pd.read_csv(master_path)
print("master:", df_master.shape)

df2 = pd.read_csv(df2_path)
if "playerId" in df2.columns and "accountId" not in df2.columns:
    df2 = df2.rename(columns={"playerId": "accountId"})
print("df2:", df2.shape)

cluster_dfs = {}
for name, path in cluster_paths.items():
    if os.path.exists(path):
        cluster_dfs[name] = pd.read_parquet(path)
        print(f"{name}_clustered:", cluster_dfs[name].shape)
    else:
        print(f"[SKIP] {name}: 없음")

# 타입 통일
for c in KEYS:
    if c in df_master.columns: df_master[c] = df_master[c].astype(str)
    if c in df2.columns: df2[c] = df2[c].astype(str)
for t in cluster_dfs.values():
    for c in KEYS:
        if c in t.columns:
            t[c] = t[c].astype(str)

print()

# =========================
# 1) merge 전략 검증(결과 기반)
# =========================
print("=== MERGE RESULT CHECK ===")
df_parquet_all = pd.concat(cluster_dfs.values(), ignore_index=True) if cluster_dfs else pd.DataFrame()

print("parquet concat shape:", df_parquet_all.shape)
print("master dup keys :", df_master.duplicated(KEYS).sum())
print("df2 dup keys    :", df2.duplicated(KEYS).sum())
print("parquet dup keys:", df_parquet_all.duplicated(KEYS).sum() if not df_parquet_all.empty else "N/A")

if not df_parquet_all.empty:
    pq_keys = set(map(tuple, df_parquet_all[KEYS].drop_duplicates().to_numpy()))
    ms_keys = set(map(tuple, df_master[KEYS].drop_duplicates().to_numpy()))
    print("parquet unique keys:", len(pq_keys))
    print("master unique keys :", len(ms_keys))
    print("parquet keys missing in master:", len(pq_keys - ms_keys))
    print("extra keys in master:", len(ms_keys - pq_keys))
    print("=> parquet 기준 left join 형태인지 점검 가능")

df2_keys = set(map(tuple, df2[KEYS].drop_duplicates().to_numpy()))
ms_keys = set(map(tuple, df_master[KEYS].drop_duplicates().to_numpy()))
print(f"master key 기준 df2 매치율: {len(ms_keys & df2_keys)/max(len(ms_keys),1):.2%}")

print()

# =========================
# 2) 컬럼 커버리지 (저장 없이 출력)
# =========================
print("=== COLUMN COVERAGE CHECK ===")
master_cols = set(df_master.columns)

def covered_in_master(col, master_cols):
    if col in master_cols:
        return True, col
    for cand in [f"{col}_df2", f"{col}_pq"]:
        if cand in master_cols:
            return True, cand
    return False, None

# df2 컬럼 커버리지
df2_cov = []
for c in sorted(df2.columns):
    ok, mapped = covered_in_master(c, master_cols)
    df2_cov.append((c, ok, mapped))
df2_cov_df = pd.DataFrame(df2_cov, columns=["df2_col", "covered", "master_col"])

print("[df2] 누락 컬럼 수:", (~df2_cov_df["covered"]).sum())
display(df2_cov_df.sort_values(["covered", "df2_col"]))

# parquet 컬럼 커버리지
for map_name, t in cluster_dfs.items():
    rows = []
    for c in sorted(t.columns):
        ok, mapped = covered_in_master(c, master_cols)
        rows.append((c, ok, mapped))
    cov_df = pd.DataFrame(rows, columns=[f"{map_name}_parquet_col", "covered", "master_col"])
    print(f"\n[{map_name}] 누락 컬럼 수:", (~cov_df["covered"]).sum())
    display(cov_df.sort_values(["covered", cov_df.columns[0]]))

print()

# =========================
# 3) 충돌 컬럼(suffix) 확인
# =========================
print("=== SUFFIX / COLLISION CHECK ===")
suffix_df2 = [c for c in df_master.columns if c.endswith("_df2")]
suffix_pq = [c for c in df_master.columns if c.endswith("_pq")]
print("suffix _df2:", len(suffix_df2))
print("suffix _pq :", len(suffix_pq))

collision_roots = sorted(set([c[:-4] for c in suffix_df2] + [c[:-3] for c in suffix_pq]))
print("collision roots:", len(collision_roots))
print(collision_roots[:100])

print()

# =========================
# 4) 맵 분석용 sanity check
# =========================
print("=== MAP / PERSONA SANITY CHECK ===")
map_col = next((c for c in ["map_display_name", "map_from_clustered", "mapName"] if c in df_master.columns), None)
persona_col = "persona" if "persona" in df_master.columns else None

print("map column:", map_col)
print("persona column:", persona_col)

if map_col:
    display(df_master[map_col].value_counts(dropna=False).to_frame("rows"))
if persona_col:
    display(df_master[persona_col].value_counts(dropna=False).to_frame("rows"))
if map_col and persona_col:
    display(pd.crosstab(df_master[map_col], df_master[persona_col], dropna=False))

print("검증 완료 ✅")

=== LOAD ===
master: (766474, 68)
df2: (828101, 31)
erangel_clustered: (213526, 37)
miramar_clustered: (177243, 31)
taego_clustered: (214455, 31)
rondo_clustered: (161250, 31)

=== MERGE RESULT CHECK ===
parquet concat shape: (766474, 38)
master dup keys : 0
df2 dup keys    : 0
parquet dup keys: 0
parquet unique keys: 766474
master unique keys : 766474
parquet keys missing in master: 0
extra keys in master: 0
=> parquet 기준 left join 형태인지 점검 가능
master key 기준 df2 매치율: 96.05%

=== COLUMN COVERAGE CHECK ===
[df2] 누락 컬럼 수: 0


,df2_col,covered,master_col
0,DBNOs,True,DBNOs
1,accountId,True,accountId
2,assists,True,assists
3,boosts,True,boosts_df2
4,createdAt,True,createdAt
5,currentRankPoint,True,currentRankPoint
6,damageDealt,True,damageDealt_df2
7,deathType,True,deathType
8,gameMode,True,gameMode
9,game_type,True,game_type



[erangel] 누락 컬럼 수: 0


,erangel_parquet_col,covered,master_col
0,accountId,True,accountId
1,altitude_variance,True,altitude_variance
2,bluezone_exposure_ratio,True,bluezone_exposure_ratio
3,boosts,True,boosts_df2
4,damageDealt,True,damageDealt_df2
5,drop_distance_from_path,True,drop_distance_from_path
6,drop_x,True,drop_x
7,drop_y,True,drop_y
8,early_enemy_density,True,early_enemy_density
9,heal_boost_use,True,heal_boost_use



[miramar] 누락 컬럼 수: 0


...


=== SUFFIX / COLLISION CHECK ===
suffix _df2: 8
suffix _pq : 8
collision roots: 8
['boosts', 'damageDealt', 'heals', 'kills', 'rideDistance', 'timeSurvived', 'walkDistance', 'winPlace']

=== MAP / PERSONA SANITY CHECK ===
map column: mapName
persona column: persona


,rows
mapName,
Tiger_Main,209069
Baltic_Main,192632
Desert_Main,174237
Neon_Main,160294
NaN,30242


,rows
persona,
⚔️ 중앙 교전형,230339
❓ 혼합형,218281
🌿 외곽 운영형,116600
🦅 게릴라 운영형,106865
🏃 중앙 점령형,94389


persona,⚔️ 중앙 교전형,❓ 혼합형,🌿 외곽 운영형,🏃 중앙 점령형,🦅 게릴라 운영형
mapName,,,,,
Baltic_Main,35103,43425,60345,20767,32992
Desert_Main,61814,53858,15122,14781,28662
Neon_Main,59178,49813,17041,15750,18512
Tiger_Main,73490,66286,20330,25124,23839
NaN,754,4899,3762,17967,2860


검증 완료 ✅


In [ ]:
<!-- import os, pandas as pd

OUT_DIR = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output"
master_path = os.path.join(OUT_DIR, "master_all_columns.csv")
df2_path    = r"C:/Users/qkrtl/Downloads/df2.csv"

cluster_paths = [
    os.path.join(OUT_DIR, "erangel_clustered.parquet"),
    os.path.join(OUT_DIR, "miramar_clustered.parquet"),
    os.path.join(OUT_DIR, "taego_clustered.parquet"),
    os.path.join(OUT_DIR, "rondo_clustered.parquet"),
]
KEYS = ["matchId", "accountId"]

# load
master = pd.read_csv(master_path)
df2 = pd.read_csv(df2_path)
if "playerId" in df2.columns and "accountId" not in df2.columns:
    df2 = df2.rename(columns={"playerId":"accountId"})

parquets = []
for p in cluster_paths:
    if os.path.exists(p):
        parquets.append(pd.read_parquet(p)[KEYS])
pq = pd.concat(parquets, ignore_index=True)

# type + strip
for d in (master, df2, pq):
    for c in KEYS:
        d[c] = d[c].astype(str).str.strip()

# counts
print("rows:", {"master":len(master), "df2":len(df2), "parquet":len(pq)})

# unique keys
ms_keys = master[KEYS].drop_duplicates()
df2_keys = df2[KEYS].drop_duplicates()
pq_keys = pq[KEYS].drop_duplicates()

print("unique keys:", {"master":len(ms_keys), "df2":len(df2_keys), "parquet":len(pq_keys)})

# duplicates (how many rows would collapse if deduped)
print("dup rows by key:", {
    "master": master.duplicated(KEYS).sum(),
    "df2": df2.duplicated(KEYS).sum(),
    "parquet": pq.duplicated(KEYS).sum()
})

# key set relationships
ms_set  = set(map(tuple, ms_keys.to_numpy()))
df2_set = set(map(tuple, df2_keys.to_numpy()))
pq_set  = set(map(tuple, pq_keys.to_numpy()))

print("df2-only keys (will be dropped if parquet is base LEFT):", len(df2_set - pq_set))
print("parquet-only keys:", len(pq_set - df2_set))
print("parquet keys missing in master:", len(pq_set - ms_set))
print("master keys not in parquet:", len(ms_set - pq_set)) -->

rows: {'master': 766474, 'df2': 828101, 'parquet': 764250}
unique keys: {'master': 766474, 'df2': 828101, 'parquet': 764250}
dup rows by key: {'master': np.int64(0), 'df2': np.int64(0), 'parquet': np.int64(0)}
df2-only keys (will be dropped if parquet is base LEFT): 91869
parquet-only keys: 28018
parquet keys missing in master: 0
master keys not in parquet: 2224


In [ ]:
# import os
# import pandas as pd
# import numpy as np

# OUT_DIR = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output"
# KEYS = ["matchId", "accountId"]

# feature_paths = [
#     os.path.join(OUT_DIR, "erangel_features.parquet"),
#     # 아래는 있으면 추가 (너 환경에 맞게)
#     os.path.join(OUT_DIR, "miramar_features.parquet"),
#     os.path.join(OUT_DIR, "taego_features.parquet"),
#     os.path.join(OUT_DIR, "rondo_features.parquet"),
# ]

# cluster_paths = [
#     os.path.join(OUT_DIR, "erangel_clustered.parquet"),
#     os.path.join(OUT_DIR, "miramar_clustered.parquet"),
#     os.path.join(OUT_DIR, "taego_clustered.parquet"),
#     os.path.join(OUT_DIR, "rondo_clustered.parquet"),
# ]

# df2_path = r"C:/Users/qkrtl/Downloads/df2.csv"

# # 1) 로드
# feat_list = []
# for p in feature_paths:
#     if os.path.exists(p):
#         t = pd.read_parquet(p)
#         t["__src_features"] = os.path.basename(p)
#         feat_list.append(t)
#         print("[features]", os.path.basename(p), t.shape)
#     else:
#         print("[features skip]", p)

# clst_list = []
# for p in cluster_paths:
#     if os.path.exists(p):
#         t = pd.read_parquet(p)
#         t["__src_clustered"] = os.path.basename(p)
#         clst_list.append(t)
#         print("[clustered]", os.path.basename(p), t.shape)
#     else:
#         print("[clustered skip]", p)

# df2 = pd.read_csv(df2_path)
# if "playerId" in df2.columns and "accountId" not in df2.columns:
#     df2 = df2.rename(columns={"playerId": "accountId"})

# df_feat = pd.concat(feat_list, ignore_index=True) if feat_list else pd.DataFrame()
# df_clst = pd.concat(clst_list, ignore_index=True) if clst_list else pd.DataFrame()

# # 2) 키 타입 통일
# for dname, d in [("df2", df2), ("features", df_feat), ("clustered", df_clst)]:
#     for c in KEYS:
#         if c in d.columns:
#             d[c] = d[c].astype(str).str.strip()
#     print(dname, "shape:", d.shape)

# # 3) key set 생성
# df2_keys   = set(map(tuple, df2[KEYS].drop_duplicates().to_numpy()))
# feat_keys  = set(map(tuple, df_feat[KEYS].drop_duplicates().to_numpy())) if not df_feat.empty else set()
# clst_keys  = set(map(tuple, df_clst[KEYS].drop_duplicates().to_numpy())) if not df_clst.empty else set()

# print("\n=== key counts ===")
# print("df2 keys     :", len(df2_keys))
# print("features keys:", len(feat_keys))
# print("clustered keys:", len(clst_keys))

# # 4) features에는 있고 clustered에는 없는 키 = 클러스터링 단계에서 탈락한 후보
# feat_not_clst = feat_keys - clst_keys
# print("\nfeatures에는 있고 clustered에는 없는 키 수:", len(feat_not_clst))

# # 5) 그 중 survival_time < 120 비율 확인
# if not df_feat.empty and "survival_time" in df_feat.columns:
#     lost_from_clustering = df_feat[df_feat[KEYS].apply(tuple, axis=1).isin(feat_not_clst)].copy()
#     print("lost_from_clustering rows:", lost_from_clustering.shape)

#     # 중복키 없다고 가정하되 혹시 몰라 dedupe
#     lost_from_clustering = lost_from_clustering.drop_duplicates(subset=KEYS)

#     if "survival_time" in lost_from_clustering.columns:
#         s = lost_from_clustering["survival_time"]
#         print("\n[survival_time stats of lost_from_clustering]")
#         print(s.describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]))

#         cond_lt120 = (s < 120)
#         cond_na    = s.isna()

#         print("\n<120 sec count :", int(cond_lt120.sum()))
#         print("<120 sec ratio  :", float(cond_lt120.mean()))
#         print("NaN count       :", int(cond_na.sum()))
#         print("NaN ratio       :", float(cond_na.mean()))
#         print(">=120 count     :", int((~cond_lt120 & ~cond_na).sum()))
#         print(">=120 ratio     :", float((~cond_lt120 & ~cond_na).mean()))
# else:
#     print("features 데이터 또는 survival_time 컬럼이 없어 체크 불가")

[features] erangel_features.parquet (234245, 29)
[features skip] C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output\miramar_features.parquet
[features skip] C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output\taego_features.parquet
[features skip] C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output\rondo_features.parquet
[clustered] erangel_clustered.parquet (213526, 38)
[clustered] miramar_clustered.parquet (176232, 32)
[clustered] taego_clustered.parquet (213857, 32)
[clustered] rondo_clustered.parquet (160635, 32)
df2 shape: (828101, 31)
features shape: (234245, 29)
clustered shape: (764250, 39)

=== key counts ===
df2 keys     : 828101
features keys: 234245
clustered keys: 764250

features에는 있고 clustered에는 없는 키 수: 20719
lost_from_clustering rows: (20719, 29)

[survival_time stats of lost_from_clustering]
count    20719.000000
mean        67.915585
std      

In [ ]:
# # df2-only keys (clustered 기준 master에서 빠질 대상)
# df2_only = df2_keys - clst_keys
# print("df2-only keys:", len(df2_only))

# # features에 있는지 / 없는지 분해
# df2_only_in_features = df2_only & feat_keys
# df2_only_not_in_features = df2_only - feat_keys

# print("df2-only & features 존재:", len(df2_only_in_features))
# print("df2-only & features에도 없음:", len(df2_only_not_in_features))

# # features에 존재하는 df2-only 키들의 survival_time 확인
# if not df_feat.empty and "survival_time" in df_feat.columns:
#     tmp = df_feat[df_feat[KEYS].apply(tuple, axis=1).isin(df2_only_in_features)].copy()
#     tmp = tmp.drop_duplicates(subset=KEYS)

#     s = tmp["survival_time"]
#     print("\n[df2-only but in features] survival_time describe")
#     print(s.describe(percentiles=[.01,.05,.1,.25,.5,.75,.9,.95,.99]))

#     print("survival_time < 120 :", int((s < 120).sum()), f"({(s < 120).mean():.2%})")
#     print("survival_time isna  :", int(s.isna().sum()),   f"({s.isna().mean():.2%})")
#     print("survival_time >=120 :", int(((s >= 120) & s.notna()).sum()), f"({((s >= 120) & s.notna()).mean():.2%})")

df2-only keys: 91869
df2-only & features 존재: 19651
df2-only & features에도 없음: 72218

[df2-only but in features] survival_time describe
count    19651.000000
mean        67.146557
std         32.642026
min          0.000000
1%           0.000000
5%          10.000000
10%         20.000000
25%         40.000000
50%         69.000000
75%         97.000000
90%        109.000000
95%        117.000000
99%        119.000000
max        119.000000
Name: survival_time, dtype: float64
survival_time < 120 : 19651 (100.00%)
survival_time isna  : 0 (0.00%)
survival_time >=120 : 0 (0.00%)


In [ ]:
# import pandas as pd
# import os

# OUT_DIR = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output"
# df2_path = r"C:/Users/qkrtl/Downloads/df2.csv"

# master_path = os.path.join(OUT_DIR, "master_all_columns.csv")
# df2 = pd.read_csv(df2_path)
# master = pd.read_csv(master_path)

# if "playerId" in df2.columns and "accountId" not in df2.columns:
#     df2 = df2.rename(columns={"playerId":"accountId"})

# for d in (df2, master):
#     d["matchId"] = d["matchId"].astype(str).str.strip()
#     d["accountId"] = d["accountId"].astype(str).str.strip()

# KEYS = ["matchId","accountId"]
# df2_keys = set(map(tuple, df2[KEYS].to_numpy()))
# ms_keys  = set(map(tuple, master[KEYS].to_numpy()))

# df2_only = df2_keys - ms_keys
# df2_only_df = df2[df2[KEYS].apply(tuple, axis=1).isin(df2_only)].copy()

# print("df2-only rows:", len(df2_only_df))

# # df2에 있는 후보 컬럼들 중 존재하는 것만 요약
# cand = ["mapName","gameMode","queueSize","teamSize","perspective","seasonState","isRanked","matchType","shardId","serverName"]
# use = [c for c in cand if c in df2_only_df.columns]
# print("available cols:", use)

# for c in use:
#     print("\n===", c, "===")
#     print(df2_only_df[c].value_counts(dropna=False).head(20))

df2-only rows: 91869
available cols: ['mapName', 'gameMode']

=== mapName ===
mapName
Tiger_Main     34555
Baltic_Main    20935
Desert_Main    18397
Neon_Main      17982
Name: count, dtype: int64

=== gameMode ===
gameMode
squad        87922
squad-fpp     3947
Name: count, dtype: int64


In [ ]:
## DF2_PATH = r"C:/Users/qkrtl/Downloads/df2.csv"                  # 예: r"C:\...\df2.csv"
## MASTER_PATH = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/master_all_columns.csv"               # 예: r"C:\...\master_all_columns.csv"



import pandas as pd
import numpy as np
import re

# =========================
# 경로 설정
# =========================
DF2_PATH = r"C:/Users/qkrtl/Downloads/df2.csv"                  # 예: r"C:\...\df2.csv"
MASTER_PATH = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/master_all_columns.csv"               # 예: r"C:\...\master_all_columns.csv"

SURVIVAL_THRESHOLD = 120

# 정규화 기준 키 후보
KEY_CANDIDATES_NORM = [
    ['matchid', 'accountid'],
    ['matchid', 'playerid'],
    ['matchid', 'playername'],
    ['matchid', 'charactername'],
    ['matchid', 'teamid', 'accountid'],
]

SURV_CANDIDATES_NORM = ['survivaltime', 'timealive']


# =========================
# 유틸
# =========================
def norm_col(c):
    c = str(c).strip().lower()
    c = re.sub(r'[\s_\-]+', '', c)
    c = re.sub(r'[^0-9a-z가-힣]', '', c)
    return c

def build_n2a(df):
    n2a = {}
    for c in df.columns:
        n = norm_col(c)
        if n not in n2a:
            n2a[n] = c
    return n2a

def pick_keys(master, df2, candidates_norm):
    m_n2a = build_n2a(master)
    d_n2a = build_n2a(df2)
    for cand in candidates_norm:
        if all(k in m_n2a for k in cand) and all(k in d_n2a for k in cand):
            m_keys = [m_n2a[k] for k in cand]
            d_keys = [d_n2a[k] for k in cand]
            return cand, m_keys, d_keys
    return None, None, None

def pick_surv(master):
    m_n2a = build_n2a(master)
    for c in SURV_CANDIDATES_NORM:
        if c in m_n2a:
            return m_n2a[c]
    return None

def normalize_key_values(df, keys):
    out = df.copy()
    for k in keys:
        out[k] = out[k].astype('string').str.strip()
        out[k] = out[k].replace('', pd.NA)
    return out

def compare_unique(master_df, df2_df, m_keys, d_keys):
    # df2 키명을 master 키명으로 맞춤
    rename_map = {d_keys[i]: m_keys[i] for i in range(len(m_keys))}
    df2_u = df2_df.rename(columns=rename_map).copy()
    master_u = master_df.copy()

    master_u = normalize_key_values(master_u, m_keys)
    df2_u = normalize_key_values(df2_u, m_keys)

    m_unique = master_u[m_keys].drop_duplicates()
    d_unique = df2_u[m_keys].drop_duplicates()

    cmp = m_unique.merge(d_unique, on=m_keys, how='outer', indicator=True)
    vc = cmp['_merge'].value_counts()

    return {
        'master_only_left_only': int(vc.get('left_only', 0)),
        'df2_only_right_only': int(vc.get('right_only', 0)),
        'both': int(vc.get('both', 0)),
        'master_unique_keys': len(m_unique),
        'df2_unique_keys': len(d_unique),
    }


# =========================
# 실행
# =========================
df2 = pd.read_csv(DF2_PATH, low_memory=False)
master = pd.read_csv(MASTER_PATH, low_memory=False)

cand_norm, m_keys, d_keys = pick_keys(master, df2, KEY_CANDIDATES_NORM)
if m_keys is None:
    print("[키 탐지 실패] 아래 공통 컬럼을 보고 KEY_CANDIDATES_NORM 수정 필요")
    m_n2a = build_n2a(master)
    d_n2a = build_n2a(df2)
    common = sorted(set(m_n2a) & set(d_n2a))
    for n in common:
        print(f"{n:<30} | master: {m_n2a[n]} | df2: {d_n2a[n]}")
    raise KeyError("공통 키 후보를 찾지 못했습니다.")

surv_col = pick_surv(master)

print(f"[선택 키(정규화)] {cand_norm}")
print(f"[master 키] {m_keys}")
print(f"[df2 키] {d_keys}")
print(f"[생존시간 컬럼] {surv_col}")

# 1) 필터 제거 전 (master 전체)
res_all = compare_unique(master, df2, m_keys, d_keys)

# 2) 필터 적용 후 (master survival_time >= 120)
if surv_col is None:
    raise KeyError("master에서 survival_time 계열 컬럼을 찾지 못했습니다.")

master_tmp = master.copy()
master_tmp[surv_col] = pd.to_numeric(master_tmp[surv_col], errors='coerce')

master_filtered = master_tmp[master_tmp[surv_col] >= SURVIVAL_THRESHOLD].copy()
res_filtered = compare_unique(master_filtered, df2, m_keys, d_keys)

# =========================
# 결과 출력
# =========================
print("\n" + "="*80)
print("[비교 결과: UNIQUE KEY 기준]")
print("- master 전체(필터 제거 전)")
for k, v in res_all.items():
    print(f"  {k:<24}: {v:,}")

print("\n- master survival_time>=120 (필터 적용 후)")
for k, v in res_filtered.items():
    print(f"  {k:<24}: {v:,}")

print("\n" + "="*80)
delta_left = res_filtered['master_only_left_only'] - res_all['master_only_left_only']
delta_both = res_filtered['both'] - res_all['both']

print("[핵심 변화량]")
print(f"  left_only 변화 (필터후 - 필터전): {delta_left:+,}")
print(f"  both 변화     (필터후 - 필터전): {delta_both:+,}")

print("\n[해석 가이드]")
print("- 필터 적용 후 left_only/right_only가 팀원이 말한 누락 규모로 맞아떨어지면 생존시간 필터 영향이 큼")
print("- 필터 제거 전에도 큰 누락이 남아 있으면 생존시간 외(키/다른 필터) 원인이 있음")

[키 탐지 실패] 아래 공통 컬럼을 보고 KEY_CANDIDATES_NORM 수정 필요
assists                        | master: assists | df2: assists
createdat                      | master: createdAt | df2: createdAt
currentrankpoint               | master: currentRankPoint | df2: currentRankPoint
dbnos                          | master: DBNOs | df2: DBNOs
deathtype                      | master: deathType | df2: deathType
gamemode                       | master: gameMode | df2: gameMode
gametype                       | master: game_type | df2: game_type
headshotkills                  | master: headshotKills | df2: headshotKills
killplace                      | master: killPlace | df2: killPlace
killstreaks                    | master: killStreaks | df2: killStreaks
longestkill                    | master: longestKill | df2: longestKill
mapname                        | master: mapName | df2: mapName
matchid                        | master: matchId | df2: matchId
name                           | master: name | df2: name
r

KeyError: '공통 키 후보를 찾지 못했습니다.'

In [ ]:
import pandas as pd
import re

DF2_PATH = r"C:/Users/qkrtl/Downloads/df2.csv"                  # 예: r"C:\...\df2.csv"
MASTER_PATH = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/master_all_columns.csv"               # 예: r"C:\...\master_all_columns.csv"


def norm_col(c):
    c = str(c).strip().lower()
    c = re.sub(r'[\s_\-]+', '', c)
    c = re.sub(r'[^0-9a-z가-힣]', '', c)
    return c

def build_n2a(df):
    n2a = {}
    for c in df.columns:
        n = norm_col(c)
        if n not in n2a:
            n2a[n] = c
    return n2a

# 헤더만 읽기 (빠름)
df2 = pd.read_csv(DF2_PATH, nrows=0)
master = pd.read_csv(MASTER_PATH, nrows=0)

d_n2a = build_n2a(df2)
m_n2a = build_n2a(master)

common = sorted(set(d_n2a.keys()) & set(m_n2a.keys()))

print("="*100)
print(f"[공통 컬럼 개수(정규화 기준)] {len(common)}")

print("\n[공통 컬럼 전체]")
for n in common:
    print(f"{n:<30} | df2: {d_n2a[n]} | master: {m_n2a[n]}")

print("\n" + "="*100)
print("[키 후보로 쓸 만한 컬럼 (id/match/player/team/name/account 포함)]")
for n in common:
    if any(tok in n for tok in ["id", "match", "player", "team", "name", "account", "character"]):
        print(f"{n:<30} | df2: {d_n2a[n]} | master: {m_n2a[n]}")

print("\n" + "="*100)
print("[df2 전용 컬럼 중 키처럼 보이는 것]")
for n, a in sorted(d_n2a.items()):
    if any(tok in n for tok in ["id", "match", "player", "team", "name", "account", "character"]):
        print(f"{n:<30} -> {a}")

print("\n" + "="*100)
print("[master 전용 컬럼 중 키처럼 보이는 것]")
for n, a in sorted(m_n2a.items()):
    if any(tok in n for tok in ["id", "match", "player", "team", "name", "account", "character"]):
        print(f"{n:<30} -> {a}")

[공통 컬럼 개수(정규화 기준)] 22

[공통 컬럼 전체]
assists                        | df2: assists | master: assists
createdat                      | df2: createdAt | master: createdAt
currentrankpoint               | df2: currentRankPoint | master: currentRankPoint
dbnos                          | df2: DBNOs | master: DBNOs
deathtype                      | df2: deathType | master: deathType
gamemode                       | df2: gameMode | master: gameMode
gametype                       | df2: game_type | master: game_type
headshotkills                  | df2: headshotKills | master: headshotKills
killplace                      | df2: killPlace | master: killPlace
killstreaks                    | df2: killStreaks | master: killStreaks
longestkill                    | df2: longestKill | master: longestKill
mapname                        | df2: mapName | master: mapName
matchid                        | df2: matchId | master: matchId
name                           | df2: name | master: name
revives         

In [ ]:
import pandas as pd
import numpy as np

# =========================
# 경로 설정
# =========================
DF2_PATH = r"C:/Users/qkrtl/Downloads/df2.csv"                  # 예: r"C:\...\df2.csv"
MASTER_PATH = r"C:/Users/qkrtl/10th/00_Project/04_final/02_parquet_file/04_temp_parquet_files/analysis_output/master_all_columns.csv"               # 예: r"C:\...\master_all_columns.csv"


# =========================
# 수동 키 지정 (지금 네 출력 기준)
# =========================
MASTER_KEYS = ["matchId", "name"]
DF2_KEYS    = ["matchId", "name"]

# master에 생존시간 컬럼 후보 (있는 것만 사용됨)
SURVIVAL_CANDIDATES = ["survival_time", "survivalTime", "timeAlive", "TimeAlive"]
SURVIVAL_THRESHOLD = 120


def normalize_keys(df, cols):
    out = df.copy()
    for c in cols:
        out[c] = out[c].astype("string").str.strip().replace("", pd.NA)
    return out

def compare_unique(master_df, df2_df, master_keys, df2_keys):
    # df2 키명을 master 키명으로 맞추기 (이 케이스는 이름 같지만 일반화)
    rename_map = {df2_keys[i]: master_keys[i] for i in range(len(master_keys))}
    d = df2_df.rename(columns=rename_map).copy()
    m = master_df.copy()

    m = normalize_keys(m, master_keys)
    d = normalize_keys(d, master_keys)

    m_u = m[master_keys].drop_duplicates()
    d_u = d[master_keys].drop_duplicates()

    cmp = m_u.merge(d_u, on=master_keys, how="outer", indicator=True)
    vc = cmp["_merge"].value_counts()

    result = {
        "master_only_left_only": int(vc.get("left_only", 0)),
        "df2_only_right_only": int(vc.get("right_only", 0)),
        "both": int(vc.get("both", 0)),
        "master_unique_keys": len(m_u),
        "df2_unique_keys": len(d_u),
    }
    return cmp, result

def pick_survival_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


# =========================
# 로드
# =========================
df2 = pd.read_csv(DF2_PATH, low_memory=False)
master = pd.read_csv(MASTER_PATH, low_memory=False)

# 컬럼 체크
for c in MASTER_KEYS:
    if c not in master.columns:
        raise KeyError(f"master에 없음: {c}")
for c in DF2_KEYS:
    if c not in df2.columns:
        raise KeyError(f"df2에 없음: {c}")

surv_col = pick_survival_col(master, SURVIVAL_CANDIDATES)
print(f"[사용 키] master={MASTER_KEYS}, df2={DF2_KEYS}")
print(f"[생존시간 컬럼] {surv_col}")

# =========================
# 1) 필터 제거 전 (master 전체)
# =========================
cmp_all, res_all = compare_unique(master, df2, MASTER_KEYS, DF2_KEYS)

# =========================
# 2) 필터 적용 후 (master survival>=120)
# =========================
if surv_col is None:
    raise KeyError(f"master에서 생존시간 컬럼을 찾지 못함. 후보={SURVIVAL_CANDIDATES}")

master2 = master.copy()
master2[surv_col] = pd.to_numeric(master2[surv_col], errors="coerce")

master_filtered = master2[master2[surv_col] >= SURVIVAL_THRESHOLD].copy()
cmp_f, res_f = compare_unique(master_filtered, df2, MASTER_KEYS, DF2_KEYS)

# =========================
# 출력
# =========================
print("\n" + "="*90)
print("[UNIQUE KEY 기준 비교 결과] (key = matchId + name)")

print("\n[1] master 전체 (생존시간 필터 제거 전)")
for k, v in res_all.items():
    print(f"  {k:<24}: {v:,}")

print("\n[2] master survival>=120 (생존시간 필터 적용 후)")
for k, v in res_f.items():
    print(f"  {k:<24}: {v:,}")

print("\n" + "="*90)
print("[변화량] (필터후 - 필터전)")
for k in ["master_only_left_only", "df2_only_right_only", "both"]:
    print(f"  {k:<24}: {res_f[k] - res_all[k]:+,}")

# 추가: 필터로 제외된 master 건수(키 기준)
m_all_u = normalize_keys(master, MASTER_KEYS)[MASTER_KEYS].drop_duplicates()
m_f_u = normalize_keys(master_filtered, MASTER_KEYS)[MASTER_KEYS].drop_duplicates()
print(f"\n[master 키 기준 필터 제외 수] {len(m_all_u) - len(m_f_u):,}")

print("\n[해석]")
print("- 필터 적용 후에만 누락이 8~9만 급으로 커지면 생존시간 필터가 주원인")
print("- 필터 제거 전에도 누락이 크면 생존시간 외 원인(다른 필터/키 정합성) 존재")

[사용 키] master=['matchId', 'name'], df2=['matchId', 'name']
[생존시간 컬럼] survival_time

[UNIQUE KEY 기준 비교 결과] (key = matchId + name)

[1] master 전체 (생존시간 필터 제거 전)
  master_only_left_only   : 1,686
  df2_only_right_only     : 91,869
  both                    : 736,232
  master_unique_keys      : 737,918
  df2_unique_keys         : 828,101

[2] master survival>=120 (생존시간 필터 적용 후)
  master_only_left_only   : 1,686
  df2_only_right_only     : 91,869
  both                    : 736,232
  master_unique_keys      : 737,918
  df2_unique_keys         : 828,101

[변화량] (필터후 - 필터전)
  master_only_left_only   : +0
  df2_only_right_only     : +0
  both                    : +0

[master 키 기준 필터 제외 수] 0

[해석]
- 필터 적용 후에만 누락이 8~9만 급으로 커지면 생존시간 필터가 주원인
- 필터 제거 전에도 누락이 크면 생존시간 외 원인(다른 필터/키 정합성) 존재
